In [1]:
%pwd
%cd /home/jovyan/work/results/Cell/
%pwd

/home/jovyan/work/results/Cell


'/home/jovyan/work/results/Cell'

In [2]:
import scanpy as sc
import pandas as pd

In [3]:
adata = sc.read_h5ad("NE_reclustered.h5ad")
adata

AnnData object with n_obs × n_vars = 115774 × 12939
    obs: 'sample_ID', 'Gender', 'INSS_stage', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'cell_type', '_scvi_batch', '_scvi_labels', 'leiden_scVI'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'INSS_stage_colors', '_scvi_manager_uuid', '_scvi_uuid', 'hvg', 'leiden_scVI', 'leiden_scVI_colors', 'leiden_scVI_sizes', 'neighbors', 'paga', 'sample_ID_colors', 'scrublet', 'umap'
    obsm: 'X_scVI', 'X_umap', '_scvi_extra_continuous_covs'
    layers: 'scvi_normalized'
    obsp: 'connectivities', 'distances'

In [4]:
adata.obs.leiden_scVI

Cell_barcode
T44_AAACCTGAGAGGTTGC.1     2
T44_AAACCTGAGAGTGAGA.1     2
T44_AAACCTGAGGCATGTG.1     2
T44_AAACCTGAGTGTACCT.1    48
T44_AAACCTGCAATGCCAT.1     2
                          ..
T40_TTTGGTTAGCAATATG.1     7
T40_TTTGGTTAGTGGAGTC.1    24
T40_TTTGGTTCATAGTAAG.1    12
T40_TTTGTCAAGTCAATAG.1     7
T40_TTTGTCATCACTTATC.1    17
Name: leiden_scVI, Length: 115774, dtype: category
Categories (56, object): ['0', '1', '2', '3', ..., '52', '53', '54', '55']

In [5]:
adata.obs[["leiden_scVI"]].value_counts()

leiden_scVI
0              5235
1              4394
2              4370
3              4097
4              4055
5              3772
6              3715
7              3628
8              3289
9              3272
10             3212
11             3188
12             3073
13             2925
14             2863
15             2784
16             2781
17             2761
18             2416
19             2407
20             2388
21             2364
22             2261
23             2205
24             2123
25             2037
26             2023
27             1912
28             1827
29             1749
30             1733
31             1665
32             1551
33             1544
34             1513
35             1435
36             1416
37             1387
38             1286
39             1240
40             1204
41             1185
42             1143
43             1127
44             1037
45              979
46              896
47              827
48              738
49      

In [6]:
adata.obs["INSS_stage"]

Cell_barcode
T44_AAACCTGAGAGGTTGC.1     I
T44_AAACCTGAGAGTGAGA.1     I
T44_AAACCTGAGGCATGTG.1     I
T44_AAACCTGAGTGTACCT.1     I
T44_AAACCTGCAATGCCAT.1     I
                          ..
T40_TTTGGTTAGCAATATG.1    IV
T40_TTTGGTTAGTGGAGTC.1    IV
T40_TTTGGTTCATAGTAAG.1    IV
T40_TTTGTCAAGTCAATAG.1    IV
T40_TTTGTCATCACTTATC.1    IV
Name: INSS_stage, Length: 115774, dtype: category
Categories (4, object): ['4S', 'I', 'III', 'IV']

start to annotate each leiden cluster with a cancer stage

In [7]:
# just take a look
adata[(adata.obs.leiden_scVI == '0') & (adata.obs.INSS_stage == 'IV')].n_obs

4926

In [8]:
data = {'4S': [adata[(adata.obs.leiden_scVI == str(i)) & (adata.obs.INSS_stage == '4S')].n_obs for i in range(len(adata.obs[["leiden_scVI"]].value_counts()))],
        'I': [adata[(adata.obs.leiden_scVI == str(i)) & (adata.obs.INSS_stage == 'I')].n_obs for i in range(len(adata.obs[["leiden_scVI"]].value_counts()))], 
        'III': [adata[(adata.obs.leiden_scVI == str(i)) & (adata.obs.INSS_stage == 'III')].n_obs for i in range(len(adata.obs[["leiden_scVI"]].value_counts()))],
        'IV': [adata[(adata.obs.leiden_scVI == str(i)) & (adata.obs.INSS_stage == 'IV')].n_obs for i in range(len(adata.obs[["leiden_scVI"]].value_counts()))],
        #'NA': [adata[(adata.obs.metacell == i) & (adata.obs.INSS_stage == 'NA')].n_obs for i in range(len(adata.obs[["metacell"]].value_counts()))]
       }

In [9]:
df = pd.DataFrame(data)
df.index.name = 'leiden'
df

,4S,I,III,IV
leiden,,,,
0,17,55,237,4926
1,15,7,50,4322
2,488,3256,6,620
3,0,2,175,3920
4,50,3571,211,223
5,140,442,48,3142
6,92,2871,644,108
7,786,1455,1009,378
8,114,2825,259,91


now normalized along each column

In [10]:
normalized_df=(df-df.min())/(df.max()-df.min())
normalized_df

,4S,I,III,IV
leiden,,,,
0,0.008057,0.015402,0.073360,1.000000
1,0.007109,0.001960,0.015232,0.877311
2,0.231280,0.911789,0.001554,0.125330
3,0.000000,0.000560,0.054088,0.795653
4,0.023697,1.000000,0.065278,0.044688
5,0.066351,0.123775,0.014610,0.637619
6,0.043602,0.803976,0.199876,0.021328
7,0.372512,0.407449,0.313335,0.076173
8,0.054028,0.791095,0.080199,0.017875


In [11]:
normalized_df.to_csv("NE_normalized_stages.csv")

In [12]:
cluster_stages = normalized_df.idxmax(axis=1)
cluster_stages.name = "discrete_cancer_stage"
cluster_stages

leiden
0      IV
1      IV
2       I
3      IV
4       I
5      IV
6       I
7       I
8       I
9     III
10     IV
11     IV
12     IV
13     4S
14     IV
15    III
16      I
17     4S
18      I
19     IV
20     4S
21    III
22    III
23    III
24    III
25     IV
26    III
27    III
28     IV
29     IV
30     IV
31      I
32     IV
33     IV
34    III
35    III
36     IV
37     IV
38     IV
39     IV
40    III
41      I
42     IV
43      I
44     4S
45    III
46      I
47    III
48      I
49      I
50     IV
51      I
52    III
53     IV
54      I
55    III
Name: discrete_cancer_stage, dtype: object

In [13]:
cluster_stages.to_csv("NE_cluster_stages.csv")